In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import gdt
import polars as pl

sys.path.insert(0, "..")
from project import check_run_settings, get_all, log, update_run_json

In [ ]:
ALL = get_all(["plants_plt"])  # LABEL_SYNONYMS below are plastid labels
# ALL = {"plants_plt": {"solanaceae": get_info("plants_plt", "solanaceae")}}
OVERWRITE = False  # True allows changing the settings of an existing cds run

GFF3_COLUMNS = [
    "seqid",
    "source",
    "type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes",
]
GENE_TYPES = {"gene", "pseudogene"}  # where the Parent chain is expected to end
UNKNOWN = "unknown"

LABEL_SYNONYMS = {
    "PLT-LHBA": "PLT-PSBZ",
    "PLT-IHBA": "PLT-PSBZ",  # typo of lhbA in the source annotation
    "PLT-YCF9": "PLT-PSBZ",
    "PLT-PBF1": "PLT-PSBN",
    "PLT-YCF6": "PLT-PETN",
    "PLT-YCF10": "PLT-CEMA",
    "PLT-YCF5": "PLT-CCSA",
    "PLT-CLPP1": "PLT-CLPP",
}

In [ ]:
def read_gff3(path: Path) -> pl.DataFrame:
    """GFF3 as a DataFrame, with ID, Parent and gene= pulled out of the attributes."""
    df = pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        new_columns=GFF3_COLUMNS,
        comment_prefix="#",
        quote_char=None,
        schema_overrides={c: pl.String for c in GFF3_COLUMNS},
    )
    attr = pl.col("attributes")
    return df.with_columns(
        pl.col("start").cast(pl.Int64),
        pl.col("end").cast(pl.Int64),
        attr.str.extract(r"(?:^|;)ID=([^;]+)").alias("ID"),
        attr.str.extract(r"(?:^|;)Parent=([^;]+)").alias("Parent"),
        attr.str.extract(r"(?:^|;)gene=([^;]+)").alias("gene"),
    )


def resolve_ancestor(
    parent: str | None, id_type: dict[str, str], id_parent: dict[str, str | None]
) -> tuple[str | None, str | None, str]:
    """Walk up the Parent chain until a gene/pseudogene. Returns (ancestor ID, ancestor type, status)."""
    if parent is None:
        return None, None, "no_parent"
    if "," in parent:
        return parent, None, "multi_parent"

    current, seen = parent, set()
    while True:
        if current not in id_type:
            return current, None, "missing_parent"
        ftype = id_type[current]
        if ftype in GENE_TYPES:
            return current, ftype, "ok"
        seen.add(current)
        nxt = id_parent.get(current)
        if nxt is None or nxt in seen or "," in nxt:
            # chain ends (or loops) without reaching a gene
            return current, ftype, "no_gene_ancestor"
        current = nxt

In [ ]:
for g, subs in ALL.items():
    gdict_file = next(iter(subs.values()))["paths"]["group_gdict"]
    gdict = gdt.read_gdict(gdict_file)
    log(f"{g}: loaded {gdict_file.name}")

    def label_of(feature_id: str | None) -> str:
        if feature_id is None:
            return UNKNOWN
        entry = gdict.get(feature_id)
        if entry is None:
            return UNKNOWN
        return LABEL_SYNONYMS.get(entry.label, entry.label)

    for subg, info in subs.items():
        p = info["paths"]
        cds_run = info["runs"]["cds"]
        run_dir = p["species_tree"] / "cds" / cds_run
        run_json = run_dir / "run.json"
        if not p["tsv"].exists():
            log(f"{g}/{subg}: no {p['tsv'].name}, run 01_get_data first; skipping")
            continue

        settings = {"label_synonyms": LABEL_SYNONYMS, "gene_types": sorted(GENE_TYPES)}
        check_run_settings(run_json, settings, OVERWRITE)
        run_dir.mkdir(parents=True, exist_ok=True)

        tsv = pl.read_csv(p["tsv"], separator="\t")
        log(f"{g}/{subg}: {tsv.height} ANs | cds run: {cds_run}")

        frames, missing_gff = [], []
        for an in tsv["AN"]:
            gff_file = p["gff3"] / f"{an}.gff3"
            if not gff_file.exists():
                missing_gff.append(an)
                continue
            df = read_gff3(gff_file)

            # lookups for this genome; first occurrence wins
            # (CDS pieces share an ID, but they are never parents)
            firsts = df.filter(pl.col("ID").is_not_null()).unique(
                "ID", keep="first", maintain_order=True
            )
            id_type = dict(zip(firsts["ID"], firsts["type"]))
            id_parent = dict(zip(firsts["ID"], firsts["Parent"]))

            ancestors = [
                resolve_ancestor(par, id_type, id_parent) for par in df["Parent"]
            ]
            df = df.with_columns(
                pl.lit(an).alias("AN"),
                pl.Series("gdt_label", [label_of(i) for i in df["ID"]]),
                pl.Series("gene_ancestor", [a[0] for a in ancestors], dtype=pl.String),
                pl.Series(
                    "gene_ancestor_type", [a[1] for a in ancestors], dtype=pl.String
                ),
                pl.Series("parent_status", [a[2] for a in ancestors]),
            ).with_columns(
                pl.when(pl.col("parent_status") == "ok")
                .then(
                    pl.col("gene_ancestor").map_elements(
                        label_of, return_dtype=pl.String
                    )
                )
                .otherwise(pl.lit(UNKNOWN))
                .alias("parent_gdt_label"),
            )
            frames.append(df)

        if missing_gff:
            log(f"  {len(missing_gff)} ANs without a GFF3: {missing_gff[:5]}...")

        all_rows = pl.concat(frames).select(
            "AN",
            "type",
            "start",
            "end",
            "strand",
            "phase",
            "ID",
            "Parent",
            "gene",
            "gdt_label",
            "gene_ancestor",
            "gene_ancestor_type",
            "parent_status",
            "parent_gdt_label",
            "attributes",
        )
        all_rows.write_csv(run_dir / "gff3_raw.tsv", separator="\t")

        # --- clean CDS: Parent chain reaches a gene that has a gdt label
        cds = all_rows.filter(pl.col("type") == "CDS")
        cds_clean = (
            cds.filter(pl.col("parent_status") == "ok")
            .filter(pl.col("parent_gdt_label") != UNKNOWN)
            .drop("phase", "type", "parent_status")
        )
        cds_clean.write_csv(run_dir / "gff3_cds.tsv", separator="\t")

        # status 'gff3_done': 02_cds_fasta must run (again) before MAFFT accepts this cds run
        update_run_json(
            run_json,
            run_name=cds_run,
            group=g,
            subgroup=subg,
            status="gff3_done",
            date_gff3=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            **settings,
            gdict=str(gdict_file.relative_to(p["group_dir"].parent)),
            gff3_dir=str(p["gff3"].relative_to(p["dir"])),
            n_ans=tsv.height,
            n_ans_without_gff3=len(missing_gff),
        )

        # --- summary
        log(f"  {all_rows.height} rows -> {run_dir.relative_to(p['dir'])}/gff3_raw.tsv")
        log(
            f"  {cds_clean.height} clean CDS rows in {cds_clean['AN'].n_unique()} ANs -> gff3_cds.tsv"
        )
        print(
            cds.group_by("parent_status", "gene_ancestor_type")
            .agg(pl.len().alias("rows"), pl.col("AN").n_unique().alias("ANs"))
            .sort("rows", descending=True)
        )
        unlabelled = cds.filter(
            (pl.col("parent_status") == "ok") & (pl.col("parent_gdt_label") == UNKNOWN)
        )
        if unlabelled.height:
            print("  genes reached by CDS but missing from the gdict:")
            print(unlabelled.select("AN", "gene_ancestor", "gene").unique().sort("AN"))

In [ ]:
cds.filter(
    (pl.col("gene_ancestor_type") != "discard-gene")
    & (
        (pl.col("parent_status") != "ok")
        | (pl.col("gene_ancestor_type") != "gene")
        | (pl.col("parent_gdt_label") == UNKNOWN)
    )
)